# Phase B0 — Google Colab CUDA preflight

This notebook is **dummy-only and recovery-unauthorized**. It verifies CUDA, frozen-resource hashes, CPU/GPU tolerance, and dummy throughput. It cannot run the scientific development experiment.


## 1. Confirm a GPU runtime
Use **Runtime → Change runtime type → T4 GPU** (or another available GPU) before running this cell.


In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then reconnect.')
print('GPU:', torch.cuda.get_device_name(0))


## 2. Upload the one preflight bundle
Upload the `dist/phase-b0-colab-preflight-<commit>.tar.gz` file created on the Mac.


In [ ]:
from google.colab import files
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.tar.gz')]
if len(archives) != 1:
    raise RuntimeError('Upload exactly one .tar.gz preflight bundle.')
ARCHIVE = archives[0]
print('Uploaded:', ARCHIVE)


## 3. Restore the exact repository and resources


In [ ]:
import json, pathlib, shutil, subprocess, tarfile
extract_root = pathlib.Path('/content/phase_b0_preflight_bundle')
repo_root = pathlib.Path('/content/latent-stroke-dynamics')
if extract_root.exists() or repo_root.exists():
    raise RuntimeError('A preflight extraction already exists; use a fresh Colab runtime.')
extract_root.mkdir()
with tarfile.open(ARCHIVE, 'r:gz') as archive:
    archive.extractall(extract_root)
manifest = json.loads((extract_root / 'bundle_manifest.json').read_text())
if manifest['recovery_authorized'] is not False:
    raise RuntimeError('Bundle unexpectedly authorizes recovery.')
subprocess.run(['git', 'clone', str(extract_root / 'repository.bundle'), str(repo_root)], check=True)
head = subprocess.check_output(['git', '-C', str(repo_root), 'rev-parse', 'HEAD'], text=True).strip()
if head != manifest['source_commit']:
    raise RuntimeError('Restored Git commit does not match bundle manifest.')
for source in (extract_root / 'resources').rglob('*'):
    if source.is_file():
        destination = repo_root / source.relative_to(extract_root / 'resources')
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
print(json.dumps(manifest, indent=2))


## 4. Install and run the complete locked test suite


In [ ]:
%cd /content/latent-stroke-dynamics
%pip install -q -e ".[dev]"


In [ ]:
!pytest -q


Expected: **120 passed**. Do not continue if any test fails.


## 5. Run the CUDA dummy-only preflight


In [ ]:
!python experiments/22_phase_b_colab_preflight.py --report /content/phase-b0-colab-preflight-report.json


## 6. Inspect and download the report


In [ ]:
report_path = pathlib.Path('/content/phase-b0-colab-preflight-report.json')
report = json.loads(report_path.read_text())
print(json.dumps(report, indent=2, sort_keys=True))
if report['status'] != 'phase_b0_colab_cuda_preflight_passed_recovery_unauthorized':
    raise RuntimeError('Preflight did not pass; do not run any development command.')
files.download(str(report_path))


Send the downloaded JSON and full test result back for review. **This pass does not authorize recovery.**
